# calculators

In [25]:
from typing import NamedTuple
from dataclasses import dataclass
from datetime import date
from recalc_engine.calcs.base import BaseCalc
from recalc_engine.sources.definitions import HierarchyLevel


class CalcCoordinate(NamedTuple):
    entity_id: int
    hierarchy_level: HierarchyLevel
    currency_code: str
    month_date: date
    stage: str | None


@dataclass(frozen=True)
class CalcPlan:
    calculator: type[BaseCalc]
    scope: tuple[CalcCoordinate, ...]

from recalc_engine.calcs.demo_calcs import CalcB, CalcA
from recalc_engine.demo_data.generator import generate_demo_table
from datetime import date

table_1 = generate_demo_table(
    table_name="table_1",
    h_ids=[1,2],
    n_rows=100
    )

table_2 = generate_demo_table(
    table_name="table_2",
    h_ids=[1,2],
    n_rows=100
    )

from recalc_engine.config.inputs import ExecutionInputs

exec_inputs = ExecutionInputs(
    table_1=table_1,
    table_2=table_2,
    results=None,
)

n_rows exceeds available combinations. Generating 54 rows instead.
n_rows exceeds available combinations. Generating 36 rows instead.


## CalcB Multiple input table

In [ ]:
bplan = CalcPlan(
    CalcB,
    (
        CalcCoordinate(1, "hierarchy_id", "CAD", date(2026, 1, 1), "Stage 1"),
        CalcCoordinate(1, "hierarchy_id", "CAD", date(2026, 2, 1), "Stage 1"),
    )
)

calcb = CalcB(
    scope=bplan.scope,
    inputs=exec_inputs,
    results=None
)

In [ ]:
import pandas as pd
dates = [date(2026, 1, 1), date(2026, 2, 1)]

pd.concat(
    [calcb.inputs.table_1,
    calcb.inputs.table_2]
).query(
    "hierarchy_id == 1 & " \
    "measure_type_code in ['measure_3', 'measure_4'] &" \
    "month_date in @dates")

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance
3,1,CAD,Stage 1,measure_3,2026-02-01,5.0
9,1,CAD,Stage 1,measure_3,2026-01-01,14.0
7,1,CAD,Stage 1,measure_4,2026-02-01,17.0
10,1,CAD,Stage 1,measure_4,2026-01-01,6.0


In [4]:
calcb.data

month_date                                            2026-01-01  2026-02-01
hierarchy_id currency_code stage   measure_type_code                        
1            CAD           Stage 1 measure_3                14.0         5.0
                                   measure_4                 6.0        17.0

In [3]:
calcb.calculate()

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance
0,1,CAD,Stage 1,measure_b,2026-01-01,20.0
1,1,CAD,Stage 1,measure_b,2026-02-01,22.0


## CalcA prev_measure

In [26]:
aplan = CalcPlan(
    CalcA,
    (
        CalcCoordinate(1, "hierarchy_id", "CAD", date(2026, 3, 1), "Stage 1"),
        CalcCoordinate(1, "hierarchy_id", "CAD", date(2026, 1, 1), "Stage 1"),
        CalcCoordinate(2, "hierarchy_id", "CAD", date(2026, 3, 1), "Stage 2"),
        CalcCoordinate(2, "hierarchy_id", "CAD", date(2026, 1, 1), "Stage 2"),
    )
)


calca = CalcA(
    scope=aplan.scope,
    inputs=exec_inputs,
    results=None
)

In [27]:
calca.inputs.table_1.query(
    "hierarchy_id in [1, 2] & " \
    "measure_type_code in ['measure_1', 'measure_2'] &" \
    "stage in ['Stage 1', 'Stage 2']"
).sort_values(by=["hierarchy_id"])

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance
0,1,CAD,Stage 1,measure_2,2026-03-01,7.0
2,1,CAD,Stage 1,measure_1,2026-02-01,5.0
50,1,CAD,Stage 1,measure_2,2026-01-01,19.0
4,1,CAD,Stage 2,measure_1,2026-01-01,1.0
41,1,CAD,Stage 2,measure_1,2026-02-01,18.0
40,1,CAD,Stage 2,measure_2,2026-01-01,15.0
14,1,CAD,Stage 2,measure_2,2026-03-01,17.0
15,1,CAD,Stage 2,measure_2,2026-02-01,7.0
38,1,CAD,Stage 1,measure_1,2026-01-01,6.0
18,1,CAD,Stage 1,measure_2,2026-02-01,11.0


In [28]:
calca.data

month_date                                            2026-01-01  2026-03-01
hierarchy_id currency_code stage   measure_type_code                        
1            CAD           Stage 1 measure_1                 6.0        16.0
                                   measure_1_prev            NaN         5.0
                                   measure_2                19.0         7.0
2            CAD           Stage 2 measure_1                19.0         5.0
                                   measure_1_prev            NaN        15.0
                                   measure_2                16.0        16.0

In [29]:
calca.calculate()

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance
0,1,CAD,Stage 1,measure_a,2026-01-01,25.0
1,2,CAD,Stage 2,measure_a,2026-01-01,35.0
2,1,CAD,Stage 1,measure_a,2026-03-01,18.0
3,2,CAD,Stage 2,measure_a,2026-03-01,6.0


# transaction & queue

In [1]:
from recalc_engine.core.transaction import Transaction
from recalc_engine.core.queues import RecalcQueues

queue = RecalcQueues(
    Transaction(f"tx_{i}") for i in range(1, 5)
    )

In [13]:
vars(queue)

{'ib': (Transaction(uuid='tx_1'), Transaction(uuid='tx_2')),
 'cb': (Transaction(uuid='tx_3'), Transaction(uuid='tx_4'))}

In [2]:
queue.ib[1].data

,hierarchy_group_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,2,CAD,Stage 3,measure_7,2026-10-01,14.0,tx_2
1,2,CAD,Stage 3,measure_6,2026-08-01,5.0,tx_2
2,1,CAD,Stage 3,measure_6,2026-08-01,6.0,tx_2
3,1,CAD,Stage 3,measure_9,2026-09-01,4.0,tx_2
4,1,CAD,Stage 2,measure_6,2026-11-01,14.0,tx_2
5,1,CAD,Stage 1,measure_6,2026-09-01,18.0,tx_2
6,2,CAD,Stage 3,measure_6,2026-05-01,5.0,tx_2
7,1,CAD,Stage 1,measure_6,2026-10-01,13.0,tx_2
8,1,CAD,Stage 2,measure_10,2026-08-01,5.0,tx_2
9,1,CAD,Stage 3,measure_6,2026-11-01,16.0,tx_2


# Compiler

## normal

In [1]:
from recalc_engine.execution.compiler import PipeCompiler
from recalc_engine.execution.pipeline import Pipeline
from recalc_engine.core.transaction import Transaction

tx1 = Transaction("tx_1")

compiler = PipeCompiler(tx1, Pipeline.PIPE)

In [2]:
list(compiler._get_triggered_coordinates(Pipeline.PIPE[0]))

[(2, 'table_1', 'CAD', datetime.date(2026, 5, 1), 'Stage 2', 0),
 (2, 'table_1', 'CAD', datetime.date(2026, 7, 1), 'Stage 2', 0),
 (2, 'table_1', 'CAD', datetime.date(2026, 5, 1), 'Stage 2', 1),
 (2, 'table_1', 'CAD', datetime.date(2026, 7, 1), 'Stage 2', 1),
 (1, 'table_1', 'CAD', datetime.date(2026, 3, 1), 'Stage 3', 0),
 (4, 'table_1', 'CAD', datetime.date(2026, 8, 1), 'Stage 1', 0)]

In [3]:
calc_plan = next(
    compiler._plan_execution()
)
calc_plan.scope

(CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 5, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 7, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 6, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 8, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 3, 1), stage='Stage 3'),
 CalcCoordinate(entity_id=4, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 8, 1), stage='Stage 1'))

## hierarchy_level_change

In [25]:
tx2 = Transaction("tx_2")

compiler2 = PipeCompiler(tx2, Pipeline.PIPE)

In [37]:
compiler2.compile().plan[0].scope

(CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 8, 1), stage='Stage 3'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 8, 1), stage='Stage 3'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 11, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 9, 1), stage='Stage 1'),
 CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 5, 1), stage='Stage 3'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 10, 1), stage='Stage 1'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_group_id', currency_code='CAD', month_date=datetime.date(2026, 11, 1), stage='

# Pipeline

In [1]:
from recalc_engine.execution.pipeline import Pipeline
from recalc_engine.core.transaction import Transaction

tx = Transaction("tx_1")
pipe = Pipeline(tx)

pipe.compile()

In [46]:
tx.data

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,1,CAD,Stage 3,measure_2,2026-01-01,15.0,tx_1
1,4,CAD,Stage 1,measure_3,2026-03-01,4.0,tx_1
2,1,CAD,Stage 1,measure_3,2026-02-01,14.0,tx_1
3,5,CAD,Stage 1,measure_3,2026-01-01,5.0,tx_1
4,2,CAD,Stage 2,measure_1,2026-02-01,11.0,tx_1


In [2]:
list(pipe.execution_plan)[0].scope

(CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 2, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=2, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 3, 1), stage='Stage 2'),
 CalcCoordinate(entity_id=1, hierarchy_level='hierarchy_id', currency_code='CAD', month_date=datetime.date(2026, 1, 1), stage='Stage 3'))

In [6]:
from recalc_engine.config.results import RecalcResults

results = RecalcResults()
pipe.execute(results)

In [33]:
results.table_r_1

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,1,CAD,Stage 3,measure_a,2026-01-01,17.0,tx_1
1,2,CAD,Stage 2,measure_a,2026-01-01,0.0,tx_1
2,1,CAD,Stage 3,measure_a,2026-02-01,0.0,tx_1
3,2,CAD,Stage 2,measure_a,2026-02-01,8.0,tx_1
4,1,CAD,Stage 3,measure_a,2026-03-01,0.0,tx_1
5,2,CAD,Stage 2,measure_a,2026-03-01,0.0,tx_1


In [2]:
from recalc_engine.config.inputs import ExecutionInputs
from recalc_engine.config.results import RecalcResults
from recalc_engine.calcs.demo_calcs import CalcE

results = RecalcResults()
inputs = ExecutionInputs.from_plan(
            pipe.transaction, pipe.execution_plan, results
        )

calc = CalcE(
    scope=list(pipe.execution_plan)[3].scope,
    inputs=inputs,
    results=results
)

In [14]:
Transaction("tx_1").data

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,1,CAD,Stage 3,measure_2,2026-01-01,15.0,tx_1
1,4,CAD,Stage 1,measure_3,2026-03-01,4.0,tx_1
2,1,CAD,Stage 1,measure_3,2026-02-01,14.0,tx_1
3,5,CAD,Stage 1,measure_3,2026-01-01,5.0,tx_1
4,2,CAD,Stage 2,measure_1,2026-02-01,11.0,tx_1


In [3]:
calc.data

month_date                                            2026-01-01  2026-02-01  \
hierarchy_id currency_code stage   measure_type_code                           
1            CAD           Stage 1 measure_1                 NaN         NaN   
                                   measure_2                 NaN         NaN   
                                   measure_3                 NaN        16.0   
                           Stage 2 measure_1                 NaN        13.0   
                                   measure_2                 NaN         8.0   
                                   measure_3                 NaN         NaN   
                           Stage 3 measure_1                 5.0         NaN   
                                   measure_2                 2.0         NaN   
                                   measure_3                 NaN         NaN   
2            CAD           Stage 1 measure_1                 NaN        19.0   
                                   measure_2                 NaN        18.0   
                                   measure_3                 NaN         NaN   
                           Stage 2 measure_1                 NaN         4.0   
                                   measure_2                 NaN        18.0   
                                   measure_3                 NaN         NaN   
                           Stage 3 measure_1                 NaN         NaN   
                                   measure_2                 NaN         NaN   
                                   measure_3                 NaN         NaN   
3            CAD           Stage 1 measure_1                 NaN        13.0   
                                   measure_2                 NaN         NaN   
                                   measure_3                 NaN         NaN   
                           Stage 2 measure_1                 NaN         5.0   
                                   measure_2                 NaN         NaN   
                                   measure_3                 NaN         NaN   
                           Stage 3 measure_1                 NaN         NaN   
                                   measure_2                 NaN         NaN   
                                   measure_3                 NaN         NaN   
4            CAD           Stage 1 measure_1                 NaN         NaN   
                                   measure_2                 NaN         NaN   
                                   measure_3                 NaN         NaN   
5            CAD           Stage 1 measure_1                 4.0         NaN   
                                   measure_2                 NaN         NaN   
                                   measure_3                 8.0         NaN   

month_date                                            2026-03-01  
hierarchy_id currency_code stage   measure_type_code              
1            CAD           Stage 1 measure_1                 NaN  
                                   measure_2                 NaN  
                                   measure_3                 NaN  
                           Stage 2 measure_1                 NaN  
                                   measure_2                 NaN  
                                   measure_3                 NaN  
                           Stage 3 measure_1                 NaN  
                                   measure_2                 NaN  
                                   measure_3                 NaN  
2            CAD           Stage 1 measure_1                 NaN  
                                   measure_2                 NaN  
                                   measure_3                 NaN  
                           Stage 2 measure_1                 NaN  
                                   measure_2                 NaN  
                                   measure_3                 NaN  
                           Stage 3 measure_1                

In [3]:
calc.calculate()

,currency_code,stage,measure_type_code,month_date,hierarchy_group_id,balance
0,CAD,Stage 1,measure_e,2026-01-01,2,12.0
1,CAD,Stage 1,measure_e,2026-02-01,1,66.0
2,CAD,Stage 1,measure_e,2026-03-01,2,36.0
3,CAD,Stage 2,measure_e,2026-02-01,1,48.0
4,CAD,Stage 3,measure_e,2026-01-01,1,7.0


# engine

In [1]:
from recalc_engine.engine import RecalcEngine

engine = RecalcEngine("tx_1").run()

In [3]:
from recalc_engine.core.transaction import Transaction
Transaction("tx_1").data

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,1,CAD,Stage 3,measure_2,2026-01-01,15.0,tx_1
1,4,CAD,Stage 1,measure_3,2026-03-01,4.0,tx_1
2,1,CAD,Stage 1,measure_3,2026-02-01,14.0,tx_1
3,5,CAD,Stage 1,measure_3,2026-01-01,5.0,tx_1
4,2,CAD,Stage 2,measure_1,2026-02-01,11.0,tx_1


In [6]:
engine.results.table_r_1.query("measure_type_code == 'measure_a'")

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,2,CAD,Stage 2,measure_a,2026-02-01,8.0,tx_1
1,2,CAD,Stage 2,measure_a,2026-03-01,0.0,tx_1
2,1,CAD,Stage 3,measure_a,2026-01-01,17.0,tx_1


In [ ]:
engine.results.table_r_1.query("measure_type_code == 'measure_b'")

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
3,4,CAD,Stage 1,measure_b,2026-03-01,14.0,tx_1
4,1,CAD,Stage 1,measure_b,2026-02-01,3.0,tx_1
5,5,CAD,Stage 1,measure_b,2026-01-01,20.0,tx_1


In [2]:
engine.results.table_r_2

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,2,CAD,Stage 2,measure_c,2026-02-01,8.0,tx_1
1,2,CAD,Stage 2,measure_c,2026-03-01,0.0,tx_1
2,1,CAD,Stage 3,measure_c,2026-01-01,17.0,tx_1
3,4,CAD,Stage 1,measure_c,2026-03-01,-14.0,tx_1
4,1,CAD,Stage 1,measure_c,2026-02-01,-3.0,tx_1
5,5,CAD,Stage 1,measure_c,2026-01-01,-20.0,tx_1


In [4]:
display(
    Transaction("tx_1").data,
    engine.results.table_r_3,
) #before groupby

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,1,CAD,Stage 3,measure_2,2026-01-01,15.0,tx_1
1,4,CAD,Stage 1,measure_3,2026-03-01,4.0,tx_1
2,1,CAD,Stage 1,measure_3,2026-02-01,14.0,tx_1
3,5,CAD,Stage 1,measure_3,2026-01-01,5.0,tx_1
4,2,CAD,Stage 2,measure_1,2026-02-01,11.0,tx_1


,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance,transaction_id
0,1,CAD,Stage 2,measure_e,2026-02-01,21.0,tx_1
1,1,CAD,Stage 3,measure_e,2026-01-01,7.0,tx_1
2,2,CAD,Stage 1,measure_e,2026-03-01,2.0,tx_1
3,1,CAD,Stage 1,measure_e,2026-02-01,16.0,tx_1
4,2,CAD,Stage 1,measure_e,2026-01-01,21.0,tx_1


In [3]:
engine.results.table_r_3

,currency_code,stage,measure_type_code,month_date,hierarchy_group_id,balance,transaction_id
0,CAD,Stage 1,measure_e,2026-01-01,1,21.0,tx_1
1,CAD,Stage 1,measure_e,2026-02-01,1,16.0,tx_1
2,CAD,Stage 1,measure_e,2026-03-01,1,2.0,tx_1
3,CAD,Stage 2,measure_e,2026-02-01,1,21.0,tx_1
4,CAD,Stage 3,measure_e,2026-01-01,1,7.0,tx_1


In [9]:
from recalc_engine.demo_data.generator import generate_demo_table

generate_demo_table("table_1", n_rows=20).query(
    "stage == 'Stage 2' & measure_type_code == 'measure_2'"
)

,hierarchy_id,currency_code,stage,measure_type_code,month_date,balance
9,2,CAD,Stage 2,measure_2,2026-01-01,18.0
18,4,CAD,Stage 2,measure_2,2026-01-01,3.0
